# Conformal prediction intervals

Calibrate prediction intervals from cross-validation errors on the point forecast using [`ConformalIntervals`](../conformal-prediction.md).

**Dataset:** day-ahead electricity prices (same panel as the [exogenous variables](exogenous-variables.ipynb) example).

**Models:** `Tafsut` with native quantiles vs the same model with `prediction_intervals`, plus `Chronos` bolt-tiny for comparison.

In [ ]:
!UV_CONSTRAINT= UV_BUILD_CONSTRAINT= uv pip install -q "foundationforecast[plot]"

In [ ]:
import pandas as pd

from foundationforecast import FoundationForecast
from foundationforecast import ConformalIntervals
from foundationforecast.models import Chronos, Tafsut

In [ ]:
DATA = "https://timecopilot.s3.amazonaws.com/public/data/electricity_price"
df = pd.read_parquet(f"{DATA}/train.parquet")
df = df.groupby("unique_id").tail(24 * 14)
df.head()

## Native quantiles (default)

In [ ]:
native = FoundationForecast(models=[Tafsut(alias="Tafsut-native", context_length=2048)])
fcst_native = native.forecast(df=df, h=24, level=[80])
fcst_native.head()

## Conformal intervals (`prediction_intervals`)

In [ ]:
tafsut_cf = Tafsut(
    alias="Tafsut-conformal",
    context_length=2048,
    prediction_intervals=ConformalIntervals(n_windows=2, method="conformal_distribution"),
)
chronos = Chronos(repo_id="amazon/chronos-bolt-tiny", alias="Chronos-bolt", context_length=512)
chronos.prediction_intervals = ConformalIntervals(n_windows=2)

conformal_ff = FoundationForecast(models=[tafsut_cf, chronos])
fcst_cf = conformal_ff.forecast(df=df, h=24, level=[80, 90])
fcst_cf.head()

In [ ]:
FoundationForecast.plot(df, fcst_native, level=[80])